In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display


# Find the repository root automatically.
ROOT = Path.cwd().resolve()

while not (ROOT / "data_labeling").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Repository root not found.")
    ROOT = ROOT.parent


NOAA_DIR = (
    ROOT
    / "data_labeling"
    / "data_labels"
    / "noaa"
)

CL_DIR = NOAA_DIR / "CL_Labels"

MASTER_LABELS = (
    NOAA_DIR
    / "labels_noaa_2010_2026_binary.csv"
)

print("Repository:", ROOT)
print("Master labels:", MASTER_LABELS)
print("Output directory:", CL_DIR)

assert MASTER_LABELS.exists()
assert CL_DIR.exists()

Repository: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-
Master labels: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/noaa/labels_noaa_2010_2026_binary.csv
Output directory: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/noaa/CL_Labels


In [2]:
data = pd.read_csv(MASTER_LABELS)

required_columns = {"label", "goes_class"}

assert required_columns.issubset(data.columns)
assert data["label"].notna().all()
assert data["goes_class"].notna().all()
assert set(data["goes_class"].unique()).issubset({0, 1})

print("Total master-label rows:", f"{len(data):,}")
print("Duplicate image paths:", data["label"].duplicated().sum())

display(data.head())

Total master-label rows: 128,117
Duplicate image paths: 0


,label,goes_class
0,2010/12/06/HMI.m2010.12.06_07.00.00.jpg,0
1,2010/12/06/HMI.m2010.12.06_08.00.00.jpg,0
2,2010/12/06/HMI.m2010.12.06_09.00.00.jpg,0
3,2010/12/06/HMI.m2010.12.06_10.00.00.jpg,0
4,2010/12/06/HMI.m2010.12.06_11.00.00.jpg,0


In [3]:
timestamp_text = data["label"].str.extract(
    r"HMI\.m(\d{4}\.\d{2}\.\d{2}_\d{2}\.\d{2}\.\d{2})"
)[0]

data["timestamp"] = pd.to_datetime(
    timestamp_text,
    format="%Y.%m.%d_%H.%M.%S",
    errors="raise",
)

data = (
    data
    .drop_duplicates("label", keep="last")
    .sort_values("timestamp")
    .reset_index(drop=True)
)

assert data["timestamp"].notna().all()
assert data["label"].is_unique
assert data["timestamp"].is_monotonic_increasing

print("First timestamp:", data["timestamp"].min())
print("Last timestamp:", data["timestamp"].max())
print("Unique images:", f"{len(data):,}")

First timestamp: 2010-12-06 07:00:00
Last timestamp: 2026-07-31 23:00:00
Unique images: 128,117


In [4]:
def select_period(start, end):
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    return data[
        data["timestamp"].between(start, end, inclusive="both")
    ].copy()


expected_existing_splits = {
    "Stage1_train.csv": (
        "2010-12-06 07:00:00",
        "2017-01-06 03:00:00",
    ),
    "Stage1_holdout.csv": (
        "2017-01-07 04:00:00",
        "2017-12-31 23:00:00",
    ),
    "Stage2_train.csv": (
        "2018-01-02 00:00:00",
        "2023-03-08 22:00:00",
    ),
    "Stage2_holdout.csv": (
        "2023-03-09 23:00:00",
        "2023-06-29 22:00:00",
    ),
}


for filename, (start, end) in expected_existing_splits.items():
    generated = select_period(start, end)[["label", "goes_class"]]
    existing = pd.read_csv(CL_DIR / filename)

    generated = generated.sort_values("label").reset_index(drop=True)
    existing = existing.sort_values("label").reset_index(drop=True)

    pd.testing.assert_frame_equal(generated, existing)

    print(f"Verified: {filename} ({len(existing):,} rows)")

Verified: Stage1_train.csv (46,638 rows)
Verified: Stage1_holdout.csv (8,230 rows)
Verified: Stage2_train.csv (44,125 rows)
Verified: Stage2_holdout.csv (2,651 rows)


In [5]:
STAGE3_TRAIN_START = "2023-07-01 00:00:00"
STAGE3_TRAIN_END = "2024-05-06 05:00:00"

STAGE3_HOLDOUT_START = "2024-05-07 06:00:00"
STAGE3_HOLDOUT_END = "2024-06-29 10:00:00"


stage3_train = select_period(
    STAGE3_TRAIN_START,
    STAGE3_TRAIN_END,
)

stage3_holdout = select_period(
    STAGE3_HOLDOUT_START,
    STAGE3_HOLDOUT_END,
)

print("Stage 3 train rows:", f"{len(stage3_train):,}")
print("Stage 3 holdout rows:", f"{len(stage3_holdout):,}")

Stage 3 train rows: 7,351
Stage 3 holdout rows: 1,273


In [6]:
def summarize(name, frame):
    fl = int(frame["goes_class"].sum())
    nf = len(frame) - fl

    return {
        "dataset": name,
        "start": frame["timestamp"].min(),
        "end": frame["timestamp"].max(),
        "samples": len(frame),
        "FL": fl,
        "NF": nf,
        "FL_percent": 100 * fl / len(frame),
    }


stage3_summary = pd.DataFrame([
    summarize("Stage 3 train", stage3_train),
    summarize("Stage 3 holdout", stage3_holdout),
])

display(stage3_summary)

,dataset,start,end,samples,FL,NF,FL_percent
0,Stage 3 train,2023-07-01 00:00:00,2024-05-06 05:00:00,7351,3707,3644,50.428513
1,Stage 3 holdout,2024-05-07 06:00:00,2024-06-29 10:00:00,1273,994,279,78.083268


In [7]:
train_end = pd.Timestamp(STAGE3_TRAIN_END)
holdout_start = pd.Timestamp(STAGE3_HOLDOUT_START)

stage3_gap = data[
    (data["timestamp"] > train_end)
    & (data["timestamp"] < holdout_start)
].copy()

print("Training ends:", train_end)
print("Holdout starts:", holdout_start)
print("Elapsed time:", holdout_start - train_end)
print("Excluded gap samples:", len(stage3_gap))

display(
    stage3_gap[
        ["label", "goes_class", "timestamp"]
    ].head()
)

Training ends: 2024-05-06 05:00:00
Holdout starts: 2024-05-07 06:00:00
Elapsed time: 1 days 01:00:00
Excluded gap samples: 24


,label,goes_class,timestamp
109092,2024/05/06/HMI.m2024.05.06_06.00.00.jpg,1,2024-05-06 06:00:00
109093,2024/05/06/HMI.m2024.05.06_07.00.00.jpg,1,2024-05-06 07:00:00
109094,2024/05/06/HMI.m2024.05.06_08.00.00.jpg,1,2024-05-06 08:00:00
109095,2024/05/06/HMI.m2024.05.06_09.00.00.jpg,1,2024-05-06 09:00:00
109096,2024/05/06/HMI.m2024.05.06_10.00.00.jpg,1,2024-05-06 10:00:00


In [8]:
train_paths = set(stage3_train["label"])
holdout_paths = set(stage3_holdout["label"])

overlap = train_paths & holdout_paths

print("Stage 3 train/holdout overlap:", len(overlap))

assert len(overlap) == 0
assert stage3_train["label"].is_unique
assert stage3_holdout["label"].is_unique
assert stage3_train["timestamp"].max() < stage3_holdout["timestamp"].min()

print("Passed: Stage 3 train and holdout are disjoint.")

Stage 3 train/holdout overlap: 0
Passed: Stage 3 train and holdout are disjoint.


In [9]:
validation = pd.read_csv(
    CL_DIR / "validation_2024_2025.csv"
)

test = pd.read_csv(
    CL_DIR / "test_2025_2026_july.csv"
)

stage3_all_paths = (
    set(stage3_train["label"])
    | set(stage3_holdout["label"])
)

validation_overlap = (
    stage3_all_paths
    & set(validation["label"])
)

test_overlap = (
    stage3_all_paths
    & set(test["label"])
)

print("Stage 3/validation overlap:", len(validation_overlap))
print("Stage 3/test overlap:", len(test_overlap))

assert len(validation_overlap) == 0
assert len(test_overlap) == 0

print("Passed: Stage 3 is disjoint from validation and test.")

Stage 3/validation overlap: 0
Stage 3/test overlap: 0
Passed: Stage 3 is disjoint from validation and test.


In [10]:
stage3_train_output = (
    stage3_train[
        ["label", "goes_class"]
    ]
    .reset_index(drop=True)
)

stage3_holdout_output = (
    stage3_holdout[
        ["label", "goes_class"]
    ]
    .reset_index(drop=True)
)

display(stage3_train_output.head())
display(stage3_holdout_output.head())


,label,goes_class
0,2023/07/01/HMI.m2023.07.01_00.00.00.jpg,1
1,2023/07/01/HMI.m2023.07.01_01.00.00.jpg,1
2,2023/07/01/HMI.m2023.07.01_02.00.00.jpg,1
3,2023/07/01/HMI.m2023.07.01_03.00.00.jpg,1
4,2023/07/01/HMI.m2023.07.01_04.00.00.jpg,1


,label,goes_class
0,2024/05/07/HMI.m2024.05.07_06.00.00.jpg,1
1,2024/05/07/HMI.m2024.05.07_07.00.00.jpg,1
2,2024/05/07/HMI.m2024.05.07_08.00.00.jpg,1
3,2024/05/07/HMI.m2024.05.07_09.00.00.jpg,1
4,2024/05/07/HMI.m2024.05.07_10.00.00.jpg,1


In [11]:
STAGE3_TRAIN_FILE = CL_DIR / "Stage3_train.csv"
STAGE3_HOLDOUT_FILE = CL_DIR / "Stage3_holdout.csv"

stage3_train_output.to_csv(
    STAGE3_TRAIN_FILE,
    index=False,
)

stage3_holdout_output.to_csv(
    STAGE3_HOLDOUT_FILE,
    index=False,
)

print("Created:", STAGE3_TRAIN_FILE)
print("Created:", STAGE3_HOLDOUT_FILE)

Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/noaa/CL_Labels/Stage3_train.csv
Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/noaa/CL_Labels/Stage3_holdout.csv


In [12]:
stage1_train = pd.read_csv(
    CL_DIR / "Stage1_train.csv"
)

stage2_train = pd.read_csv(
    CL_DIR / "Stage2_train.csv"
)

stage3_train_saved = pd.read_csv(
    CL_DIR / "Stage3_train.csv"
)


joint_train = pd.concat(
    [
        stage1_train,
        stage2_train,
        stage3_train_saved,
    ],
    ignore_index=True,
)

print("Joint rows before duplicate removal:", f"{len(joint_train):,}")
print("Duplicate image paths:", joint_train["label"].duplicated().sum())

assert joint_train["label"].duplicated().sum() == 0
assert set(joint_train["goes_class"].unique()).issubset({0, 1})

display(joint_train.head())

Joint rows before duplicate removal: 98,114
Duplicate image paths: 0


,label,goes_class
0,2010/12/06/HMI.m2010.12.06_07.00.00.jpg,0
1,2010/12/06/HMI.m2010.12.06_08.00.00.jpg,0
2,2010/12/06/HMI.m2010.12.06_09.00.00.jpg,0
3,2010/12/06/HMI.m2010.12.06_10.00.00.jpg,0
4,2010/12/06/HMI.m2010.12.06_11.00.00.jpg,0


In [13]:
joint_fl = int(joint_train["goes_class"].sum())
joint_nf = len(joint_train) - joint_fl

joint_inverse_weight = joint_nf / joint_fl
joint_sqrt_weight = joint_inverse_weight ** 0.5

joint_summary = pd.DataFrame([{
    "dataset": "Joint train",
    "samples": len(joint_train),
    "FL": joint_fl,
    "NF": joint_nf,
    "FL_percent": 100 * joint_fl / len(joint_train),
    "inverse_FL_weight": joint_inverse_weight,
    "sqrt_inverse_FL_weight": joint_sqrt_weight,
}])

display(joint_summary)

,dataset,samples,FL,NF,FL_percent,inverse_FL_weight,sqrt_inverse_FL_weight
0,Joint train,98114,16111,82003,16.420694,5.089876,2.256075


In [14]:
JOINT_TRAIN_FILE = CL_DIR / "Joint_train.csv"

joint_train.to_csv(
    JOINT_TRAIN_FILE,
    index=False,
)

print("Created:", JOINT_TRAIN_FILE)

Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/noaa/CL_Labels/Joint_train.csv


In [15]:
files_to_verify = {
    "Stage3_train.csv": 7351,
    "Stage3_holdout.csv": 1273,
    "Joint_train.csv": 98114,
}

for filename, expected_rows in files_to_verify.items():
    path = CL_DIR / filename
    frame = pd.read_csv(path)

    assert path.exists()
    assert list(frame.columns) == ["label", "goes_class"]
    assert len(frame) == expected_rows
    assert frame["label"].is_unique
    assert frame["goes_class"].isin([0, 1]).all()

    print(
        f"Passed: {filename} | "
        f"rows={len(frame):,} | "
        f"FL={int(frame['goes_class'].sum()):,}"
    )

Passed: Stage3_train.csv | rows=7,351 | FL=3,707
Passed: Stage3_holdout.csv | rows=1,273 | FL=994
Passed: Joint_train.csv | rows=98,114 | FL=16,111
